# Bench: `get_validator_chain_commits` vs `get_chain_commits`

Both helpers return `[(WorkerChainCommit | None, Neuron), ...]`. The original `get_chain_commits` pulls every commit in the subnet via `subtensor.get_all_commitments(netuid=)` and then classifies each one as validator/miner; the new `get_validator_chain_commits` pulls only commits for hotkeys returned by the owner's validator-whitelist API, via per-UID `subtensor.get_commitment(netuid, uid)`.

On a subnet with ~10 validators out of ~256 UIDs, the new path issues ~10 small RPCs instead of one large one, and skips the role-gating logic entirely.

This notebook hits the live archive node. Re-run if the result depends on the current head.

In [ ]:
from __future__ import annotations

import time

import bittensor

from connito.shared.config import ValidatorConfig
from connito.shared.chain import (
    get_chain_commits,
    get_validator_chain_commits,
    ValidatorChainCommit,
    MinerChainCommit,
)

config = ValidatorConfig()
print('netuid =', config.chain.netuid)
print('network =', config.chain.network)
print('owner_url =', config.cycle.owner_url)

In [ ]:
subtensor = bittensor.Subtensor(network=config.chain.network)
print('connected to', config.chain.network, '@ head block', subtensor.block)

## Pick a block to query

Default: current head. Set `query_block` to an integer to compare at a specific historical block (e.g. the end of a previous `ValidatorCommit2` phase, matching the call from `build_chain_checkpoints_from_previous_phase`).

In [ ]:
query_block: int | None = None  # None = head; or set explicitly e.g. 8249652
print('querying block =', query_block if query_block is not None else 'head')

## Run the legacy `get_chain_commits`

In [ ]:
t0 = time.monotonic()
legacy = get_chain_commits(config, subtensor, block=query_block)
elapsed_legacy = time.monotonic() - t0

legacy_by_hotkey = {neuron.hotkey: (commit, neuron) for commit, neuron in legacy if commit is not None}
legacy_validators = {hk: pair for hk, pair in legacy_by_hotkey.items() if isinstance(pair[0], ValidatorChainCommit)}
legacy_miners = {hk: pair for hk, pair in legacy_by_hotkey.items() if isinstance(pair[0], MinerChainCommit)}

print(f'legacy: {elapsed_legacy:.2f}s, {len(legacy)} raw entries '
      f'({len(legacy_validators)} validator commits, {len(legacy_miners)} miner commits)')

## Run the new `get_validator_chain_commits`

In [ ]:
t0 = time.monotonic()
new = get_validator_chain_commits(config, subtensor, block=query_block)
elapsed_new = time.monotonic() - t0

new_by_hotkey = {neuron.hotkey: (commit, neuron) for commit, neuron in new if commit is not None}

print(f'new: {elapsed_new:.2f}s, {len(new)} entries returned, {len(new_by_hotkey)} validator commits parsed')
if elapsed_legacy > 0:
    print(f'speedup: {elapsed_legacy / max(elapsed_new, 1e-6):.2f}x')

## Equivalence check

The new path should return the same set of validator commits as the legacy path. The legacy path *also* returns miner commits — that's the extra work we're shedding. If a hotkey appears in `legacy_validators` but not in `new_by_hotkey`, either the whitelist API is stale or the per-UID `get_commitment` call returned empty for that UID.

In [ ]:
in_legacy_only = set(legacy_validators) - set(new_by_hotkey)
in_new_only = set(new_by_hotkey) - set(legacy_validators)
in_both = set(legacy_validators) & set(new_by_hotkey)

print(f'in both:        {len(in_both):>3} hotkeys')
print(f'legacy only:    {len(in_legacy_only):>3} hotkeys  -> {sorted(in_legacy_only)[:5]}')
print(f'new only:       {len(in_new_only):>3} hotkeys  -> {sorted(in_new_only)[:5]}')

# Spot-check: same parsed model_hash for overlapping hotkeys.
mismatches = []
for hk in in_both:
    lc, _ = legacy_validators[hk]
    nc, _ = new_by_hotkey[hk]
    if getattr(lc, 'model_hash', None) != getattr(nc, 'model_hash', None):
        mismatches.append(hk)
print(f'model_hash mismatches across overlapping hotkeys: {len(mismatches)}')

## Optional: `signature_commit=True` variant

The peer-sync path runs both flavours back-to-back (one at `validator_commit_1_end_block` with `signature_commit=True`, one at `validator_commit_2_end_block`). Same equivalence check, but with the signed-hash payload.

In [ ]:
t0 = time.monotonic()
legacy_sig = get_chain_commits(config, subtensor, block=query_block, signature_commit=True)
elapsed_legacy_sig = time.monotonic() - t0

t0 = time.monotonic()
new_sig = get_validator_chain_commits(config, subtensor, block=query_block, signature_commit=True)
elapsed_new_sig = time.monotonic() - t0

print(f'signature_commit=True   legacy: {elapsed_legacy_sig:.2f}s   new: {elapsed_new_sig:.2f}s   speedup: {elapsed_legacy_sig / max(elapsed_new_sig, 1e-6):.2f}x')

## Optional: reuse a pre-fetched lite metagraph

If the caller already has a metagraph (the validator loop does), pass it in to skip the metagraph RPC inside the helper.

In [ ]:
metagraph = subtensor.metagraph(netuid=config.chain.netuid, lite=True)

t0 = time.monotonic()
new_with_mg = get_validator_chain_commits(config, subtensor, block=query_block, metagraph=metagraph)
elapsed_with_mg = time.monotonic() - t0

print(f'with passed-in metagraph: {elapsed_with_mg:.2f}s ({len(new_with_mg)} entries)')